In [5]:
import pandas as pd
import numpy as np
import re
import joblib
from scipy.stats import entropy
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from scipy.sparse import hstack
from skl2onnx import convert_sklearn
from skl2onnx.common.data_types import FloatTensorType

In [6]:
def calculate_scipy_entropy(text):
    if not text or not isinstance(text, str): return 0
    probs = pd.Series(list(text)).value_counts() / len(text)
    return entropy(probs, base=2)

def special_char_ratio(text):
    if not text or not isinstance(text, str): return 0
    return len(re.findall(r'[^a-zA-Z0-9]', text)) / len(text)

In [7]:
df = pd.read_csv('csic_database.csv')
print("Loading dataset and enforcing label mapping...")

# Find the value that appears most often (Normal traffic)
majority_class = df['classification'].value_counts().idxmax()

# Force the majority class to be 0 (Normal) and the minority class to be 1 (Anomalous)
df['Type'] = np.where(df['classification'] == majority_class, 0, 1)

# Combine URL and content for complete context
df['full_request'] = df['URL'].fillna('') + df['content'].fillna('')

Loading dataset and enforcing label mapping...


In [8]:
numeric_features = pd.DataFrame()
numeric_features['length'] = df['full_request'].apply(len)
numeric_features['entropy'] = df['full_request'].apply(calculate_scipy_entropy)
numeric_features['spec_char_ratio'] = df['full_request'].apply(special_char_ratio)

In [9]:
X_train_num, X_test_num, y_train, y_test, raw_train, raw_test = train_test_split(
    numeric_features, df['Type'], df['full_request'], 
    test_size=0.20, stratify=df['Type'], random_state=42
)

In [10]:
vectorizer = TfidfVectorizer(analyzer='char', ngram_range=(1, 3), max_features=1000)
X_train_tfidf = vectorizer.fit_transform(raw_train)
X_test_tfidf = vectorizer.transform(raw_test)

In [11]:
scaler = StandardScaler()
X_train_final = hstack([scaler.fit_transform(X_train_num), X_train_tfidf])
X_test_final = hstack([scaler.transform(X_test_num), X_test_tfidf])

In [12]:
model = RandomForestClassifier(
    n_estimators=150, max_depth=25, min_samples_leaf=2,
    class_weight={0: 1, 1: 2},
    random_state=42, n_jobs=-1
)
model.fit(X_train_final, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",150
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",25
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",2
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric(y

In [13]:
print("\n--- Final Model Performance Report ---")
y_pred = model.predict(X_test_final)
print(classification_report(y_test, y_pred, target_names=['Normal', 'Anomalous']))
accuracy = (y_test == y_pred).mean()
print(f"Combined Accuracy: {accuracy:.4f}\n")


--- Final Model Performance Report ---
              precision    recall  f1-score   support

      Normal       0.99      0.99      0.99      7200
   Anomalous       0.99      0.98      0.98      5013

    accuracy                           0.99     12213
   macro avg       0.99      0.99      0.99     12213
weighted avg       0.99      0.99      0.99     12213

Combined Accuracy: 0.9862



In [14]:
print("Exporting pipeline for the C++ Interceptor...")
joblib.dump(vectorizer, 'tfidf_vectorizer.pkl')
joblib.dump(scaler, 'standard_scaler.pkl')

feature_count = X_train_final.shape[1]
initial_type = [('float_input', FloatTensorType([None, feature_count]))]
onnx_model = convert_sklearn(model, initial_types=initial_type)

with open("waf_brain_v1.onnx", "wb") as f:
    f.write(onnx_model.SerializeToString())

print("Success. Files generated: 'tfidf_vectorizer.pkl', 'standard_scaler.pkl', 'waf_brain_v1.onnx'")

Exporting pipeline for the C++ Interceptor...
Success. Files generated: 'tfidf_vectorizer.pkl', 'standard_scaler.pkl', 'waf_brain_v1.onnx'


In [15]:
import json
import joblib

print("Loading saved preprocessors from memory...")

# 1. Load the vectorizer and scaler
vectorizer = joblib.load('tfidf_vectorizer.pkl')
scaler = joblib.load('standard_scaler.pkl')

print("Preprocessors loaded. Cleaning NumPy types and exporting to JSON...")

# 2. Clean the vocabulary dictionary (Convert np.int64 to standard int)
clean_vocabulary = {key: int(value) for key, value in vectorizer.vocabulary_.items()}

# 3. Export TF-IDF Configuration
tfidf_config = {
    "vocabulary": clean_vocabulary,
    "idf": vectorizer.idf_.tolist(),
    "max_features": int(vectorizer.max_features) if vectorizer.max_features else None,
    "ngram_range": list(vectorizer.ngram_range)
}

with open("tfidf_config.json", "w") as f:
    json.dump(tfidf_config, f, indent=4)

# 4. Export Standard Scaler Configuration
scaler_config = {
    "mean": scaler.mean_.tolist(),
    "scale": scaler.scale_.tolist()
}

with open("scaler_config.json", "w") as f:
    json.dump(scaler_config, f, indent=4)

print("Success. The .json files have been generated without serialization errors.")

Loading saved preprocessors from memory...
Preprocessors loaded. Cleaning NumPy types and exporting to JSON...
Success. The .json files have been generated without serialization errors.


In [16]:
print(df['classification'].value_counts())
print(df['Type'].value_counts())
# Confirm 0=Normal is actually the smaller class — if not, your mapping is inverted

classification
0    36000
1    25065
Name: count, dtype: int64
Type
0    36000
1    25065
Name: count, dtype: int64


In [17]:
import onnxruntime as rt
sess = rt.InferenceSession("waf_brain_v1.onnx", providers=['CPUExecutionProvider'])
input_name = sess.get_inputs()[0].name
label_name = sess.get_outputs()[0].name

# Test a guaranteed normal from training data
sample = df[df['Type'] == 0]['full_request'].iloc[0]
num = np.array([[len(sample), calculate_scipy_entropy(sample), special_char_ratio(sample)]])
feat = hstack([scaler.transform(num), vectorizer.transform([sample])]).toarray().astype(np.float32)
pred = sess.run([label_name], {input_name: feat})[0][0]
print("Sanity check:", "NORMAL" if pred == 0 else "BUG: flagged as ANOMALOUS")

Sanity check: NORMAL


c:\Users\legoc\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


In [18]:
import joblib
import numpy as np
import pandas as pd
import re
import onnxruntime as rt
from scipy.stats import entropy
from scipy.sparse import hstack

print("1. Initializing Environment...")

# --- A. Define Utility Functions ---
def calculate_scipy_entropy(text):
    if not text or not isinstance(text, str): return 0
    probs = pd.Series(list(text)).value_counts() / len(text)
    return entropy(probs, base=2)

def special_char_ratio(text):
    if not text or not isinstance(text, str): return 0
    return len(re.findall(r'[^a-zA-Z0-9]', text)) / len(text)

# --- B. Load Preprocessors and ONNX Model ---
try:
    vectorizer = joblib.load('tfidf_vectorizer.pkl')
    scaler = joblib.load('standard_scaler.pkl')
    
    # This is the line that creates 'sess'. We wrap it in a try-block to catch missing files.
    sess = rt.InferenceSession("waf_brain_v1.onnx", providers=['CPUExecutionProvider'])
    input_name = sess.get_inputs()[0].name
    label_name = sess.get_outputs()[0].name
    print("Preprocessors and ONNX Engine loaded successfully.\n")
except FileNotFoundError as e:
    print(f"\nCRITICAL ERROR: {e}")
    print("Please ensure 'waf_brain_v1.onnx', 'tfidf_vectorizer.pkl', and 'standard_scaler.pkl' are in your current directory.")
    raise

# --- C. Use ACTUAL CSIC data instead of hand-written URLs ---
print("2. Testing with ACTUAL CSIC Dataset (not hand-written URLs)...\n")

# Pull real samples from the training data
test_cases = []
# 3 normal samples
test_cases.extend([
    (df[df['Type'] == 0]['full_request'].iloc[i], "NORMAL", "NORMAL")
    for i in range(3)
])
# 3 anomalous samples
test_cases.extend([
    (df[df['Type'] == 1]['full_request'].iloc[i], "ANOMALOUS", "ANOMALOUS")
    for i in range(3)
])

# --- D. Execute Inference Loop ---
correct = 0
for i, (raw_request, expected, label) in enumerate(test_cases):
    # Extract numerical features
    length = len(raw_request)
    ent = calculate_scipy_entropy(raw_request)
    ratio = special_char_ratio(raw_request)
    num_features = np.array([[length, ent, ratio]])
    
    # Scale numericals and extract TF-IDF
    num_scaled = scaler.transform(num_features)
    tfidf_features = vectorizer.transform([raw_request])
    
    # Combine into the exact matrix shape expected by ONNX
    final_features = hstack([num_scaled, tfidf_features])
    input_data = final_features.toarray().astype(np.float32)
    
    # Run prediction
    prediction = sess.run([label_name], {input_name: input_data})[0]
    
    # Output the result
    pred_label = "ANOMALOUS" if prediction[0] == 1 else "NORMAL"
    match = "✅" if pred_label == expected else "❌"
    if pred_label == expected:
        correct += 1
    
    print(f"Test Case {i+1}: {pred_label} (expected {expected}) {match}")
    print(f"  Payload: {raw_request[:70]}...\n")

print(f"\nAccuracy: {correct}/{len(test_cases)} ({100*correct/len(test_cases):.0f}%)")


1. Initializing Environment...
Preprocessors and ONNX Engine loaded successfully.

2. Testing with ACTUAL CSIC Dataset (not hand-written URLs)...

Test Case 1: NORMAL (expected NORMAL) ✅
  Payload: http://localhost:8080/tienda1/index.jsp HTTP/1.1...

Test Case 2: NORMAL (expected NORMAL) ✅
  Payload: http://localhost:8080/tienda1/publico/anadir.jsp?id=3&nombre=Vino+Rioj...

Test Case 3: NORMAL (expected NORMAL) ✅
  Payload: http://localhost:8080/tienda1/publico/anadir.jsp HTTP/1.1id=3&nombre=V...

Test Case 4: ANOMALOUS (expected ANOMALOUS) ✅
  Payload: http://localhost:8080/tienda1/publico/anadir.jsp?id=2&nombre=Jam%F3n+I...

Test Case 5: ANOMALOUS (expected ANOMALOUS) ✅
  Payload: http://localhost:8080/tienda1/publico/anadir.jsp HTTP/1.1id=2&nombre=J...

Test Case 6: ANOMALOUS (expected ANOMALOUS) ✅
  Payload: http://localhost:8080/tienda1/publico/anadir.jsp?id=2%2F&nombre=Jam%F3...


Accuracy: 6/6 (100%)


c:\Users\legoc\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
c:\Users\legoc\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
c:\Users\legoc\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
c:\Users\legoc\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
c:\Users\legoc\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2691: U

In [19]:

# Analyze training distribution statistics
print("\n=== TRAINING DATA STATISTICS ===")
df['length'] = df['full_request'].apply(len)
df['entropy'] = df['full_request'].apply(calculate_scipy_entropy)
df['spec_chars'] = df['full_request'].apply(special_char_ratio)

for label, name in [(0, "NORMAL"), (1, "ANOMALOUS")]:
    subset = df[df['Type'] == label]
    print(f"\n{name} (Type={label}):")
    print(f"  Length: mean={subset['length'].mean():.1f}, std={subset['length'].std():.1f}, min={subset['length'].min()}, max={subset['length'].max()}")
    print(f"  Entropy: mean={subset['entropy'].mean():.3f}, std={subset['entropy'].std():.3f}")
    print(f"  Spec chars: mean={subset['spec_chars'].mean():.3f}, std={subset['spec_chars'].std():.3f}")

print("\nYour test cases are NOT in this distribution!")
print("→ This is why the model flags them all as anomalous")



=== TRAINING DATA STATISTICS ===

NORMAL (Type=0):
  Length: mean=99.3, std=78.6, min=48, max=367
  Entropy: mean=4.659, std=0.231
  Spec chars: mean=0.182, std=0.017

ANOMALOUS (Type=1):
  Length: mean=155.2, std=110.4, min=31, max=895
  Entropy: mean=4.884, std=0.275
  Spec chars: mean=0.180, std=0.026

Your test cases are NOT in this distribution!
→ This is why the model flags them all as anomalous


In [20]:
print("\n=== DEBUG: Feature Distribution Comparison ===")
print("Training data (Type=0, Normal) sample:")
train_sample = df[df['Type'] == 0]['full_request'].iloc[0]
print(f"  Length: {len(train_sample)}, Entropy: {calculate_scipy_entropy(train_sample):.3f}, Spec chars: {special_char_ratio(train_sample):.3f}")
print(f"  First 100 chars: {train_sample[:100]}\n")

print("Test case 1 (expected normal):")
tc1 = test_cases[0]
print(f"  Length: {len(tc1)}, Entropy: {calculate_scipy_entropy(tc1):.3f}, Spec chars: {special_char_ratio(tc1):.3f}")
print(f"  First 100 chars: {tc1[:100]}")
print("  → Much shorter! Model sees this as unusual.\n")



=== DEBUG: Feature Distribution Comparison ===
Training data (Type=0, Normal) sample:
  Length: 48, Entropy: 4.452, Spec chars: 0.208
  First 100 chars: http://localhost:8080/tienda1/index.jsp HTTP/1.1

Test case 1 (expected normal):
  Length: 3, Entropy: 0.000, Spec chars: 0.000
  First 100 chars: ('http://localhost:8080/tienda1/index.jsp HTTP/1.1', 'NORMAL', 'NORMAL')
  → Much shorter! Model sees this as unusual.



In [21]:
print("=== WAF Training Data Viewer ===")

# Force pandas to display the entire string without truncation
pd.set_option('display.max_colwidth', None)

print("\n--- SAMPLE: NORMAL TRAFFIC (Class 0) ---")
# Extract 3 random normal requests
normal_samples = df[df['Type'] == 0]['full_request'].sample(n=3, random_state=42)
for i, request in enumerate(normal_samples, 1):
    print(f"\n[Normal Request {i}]:\n{request}")

print("\n" + "="*50 + "\n")

print("--- SAMPLE: ANOMALOUS TRAFFIC (Class 1) ---")
# Extract 3 random attack requests
attack_samples = df[df['Type'] == 1]['full_request'].sample(n=3, random_state=42)
for i, request in enumerate(attack_samples, 1):
    print(f"\n[Attack Payload {i}]:\n{request}")

=== WAF Training Data Viewer ===

--- SAMPLE: NORMAL TRAFFIC (Class 0) ---

[Normal Request 1]:
http://localhost:8080/tienda1/publico/entrar.jsp HTTP/1.1errorMsg=Credenciales+incorrectas

[Normal Request 2]:
http://localhost:8080/tienda1/imagenes/nuestratierra.jpg HTTP/1.1

[Normal Request 3]:
http://localhost:8080/tienda1/miembros/imagenes/zarauz.jpg HTTP/1.1


--- SAMPLE: ANOMALOUS TRAFFIC (Class 1) ---

[Attack Payload 1]:
http://localhost:8080/tienda1/miembros/editar.jsp?modo=registro&login=chi-yin&password=zAfIo&nombre=Amael&apellidos=Casselle+Cullell&email=vacey%40madrid2012.kw&dni=70541156H&direccion=Plza.+Agueda+Diez%2C+25+10%3FB&ciudad=Oria&cp=19415&provincia=Cantabria&ntc=%2520&B1=Registrar HTTP/1.1

[Attack Payload 2]:
http://localhost:8080/tienda1/publico/anadir.jsp?id=2%2F&nombre=Jam%F3n+Ib%E9rico&precio=39&cantidad=81&B1=A%F1adir+al+carrito HTTP/1.1

[Attack Payload 3]:
http://localhost:8080/tienda1/miembros/editar.jsp?modo=registro&login=teodoro&password=b9h%FA4a&nombre=

In [22]:
df['full_request'].head()

0                                                                                  http://localhost:8080/tienda1/index.jsp HTTP/1.1
1    http://localhost:8080/tienda1/publico/anadir.jsp?id=3&nombre=Vino+Rioja&precio=100&cantidad=55&B1=A%F1adir+al+carrito HTTP/1.1
2     http://localhost:8080/tienda1/publico/anadir.jsp HTTP/1.1id=3&nombre=Vino+Rioja&precio=100&cantidad=55&B1=A%F1adir+al+carrito
3     http://localhost:8080/tienda1/publico/autenticar.jsp?modo=entrar&login=choong&pwd=d1se3ci%F3n&remember=off&B1=Entrar HTTP/1.1
4      http://localhost:8080/tienda1/publico/autenticar.jsp HTTP/1.1modo=entrar&login=choong&pwd=d1se3ci%F3n&remember=off&B1=Entrar
Name: full_request, dtype: object

In [23]:
test_request = "http://localhost:8080/tienda1/publico/entrar.jsp HTTP/1.1errorMsg=Credenciales+incorrectas"

print("=== TESTING SPECIFIC HTTP REQUEST ===\n")
print(f"Request: {test_request}\n")

# Extract features
length = len(test_request)
ent = calculate_scipy_entropy(test_request)
ratio = special_char_ratio(test_request)

print(f"Feature Analysis:")
print(f"  Length: {length}")
print(f"  Entropy: {ent:.4f}")
print(f"  Special char ratio: {ratio:.4f}\n")

# Scale and vectorize
num_features = np.array([[length, ent, ratio]])
num_scaled = scaler.transform(num_features)
tfidf_features = vectorizer.transform([test_request])
final_features = hstack([num_scaled, tfidf_features])
input_data = final_features.toarray().astype(np.float32)

# Run inference
prediction = sess.run([label_name], {input_name: input_data})[0]
pred_label = "ANOMALOUS" if prediction[0] == 1 else "NORMAL"

print(f"MODEL PREDICTION: {pred_label}")
print(f"Confidence score: {prediction[0]}")


=== TESTING SPECIFIC HTTP REQUEST ===

Request: http://localhost:8080/tienda1/publico/entrar.jsp HTTP/1.1errorMsg=Credenciales+incorrectas

Feature Analysis:
  Length: 90
  Entropy: 4.6183
  Special char ratio: 0.1444

MODEL PREDICTION: NORMAL
Confidence score: 0


c:\Users\legoc\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


In [24]:

# Better test with actual confidence scores
print("\n=== IMPROVED TEST WITH CONFIDENCE SCORES ===\n")
print(f"Request: {test_request}\n")

# Run inference with both outputs
label_output, prob_output = sess.run(['output_label', 'output_probability'], {input_name: input_data})
predicted_class = label_output[0]
confidence_dict = prob_output[0]  # Returns {class: probability, ...}

pred_label = "ANOMALOUS" if predicted_class == 1 else "NORMAL"
confidence = confidence_dict.get(predicted_class, 0) * 100

print(f"Prediction: {pred_label}")
print(f"Confidence: {confidence:.2f}%")
print(f"\nDetailed Probabilities:")
print(f"  NORMAL (0):     {confidence_dict.get(0, 0)*100:.2f}%")
print(f"  ANOMALOUS (1):  {confidence_dict.get(1, 0)*100:.2f}%")



=== IMPROVED TEST WITH CONFIDENCE SCORES ===

Request: http://localhost:8080/tienda1/publico/entrar.jsp HTTP/1.1errorMsg=Credenciales+incorrectas

Prediction: NORMAL
Confidence: 92.93%

Detailed Probabilities:
  NORMAL (0):     92.93%
  ANOMALOUS (1):  7.07%


In [25]:
print(repr(df['full_request'].iloc[0]))


'http://localhost:8080/tienda1/index.jsp HTTP/1.1'
